# Cluster-Level Markov Chain Model for EcoBici Stations

This notebook models EcoBici station interactions as **Markov chains within clusters**.  
For each cluster we compute:

1. **Internal transition matrix** — probability of a bike moving from station $i$ to station $j$ within the same cluster  
2. **Outflow vector** — probability that a bike leaves the cluster (and to which external cluster)  
3. **Inflow vector** — how many bikes arrive from each external cluster  

Clustering is fully configurable: swap the `build_clusters()` function to use any method (K-Means, manual, DBSCAN, alcaldía-based, etc.).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import folium
from branca.colormap import LinearColormap

plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                         "#e87ba4", "#008300", "#6250d6", "#e34948"]),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8,
    "lines.linewidth": 2,
    "lines.markersize": 5,
})

## 1. Load and clean data

In [ ]:
def fix_station_id(station):
    """Normalise station IDs: strip whitespace, handle 'NNN-M' duplicates, return int."""
    if pd.isna(station):
        return None
    station = str(station).strip()
    if station == "":
        return None
    if "-" in station:
        station = station.split("-")[0]
    try:
        return int(float(station))
    except ValueError:
        return None


# ── Stations ──────────────────────────────────────────────────────────
stations = pd.read_csv("data/estaciones_ecobici.csv", encoding="utf-8-sig")
stations["num_cicloe"] = stations["num_cicloe"].apply(fix_station_id)
stations = stations.dropna(subset=["num_cicloe", "latitud", "longitud"])
stations["num_cicloe"] = stations["num_cicloe"].astype(int)
stations = stations.drop_duplicates(subset="num_cicloe").set_index("num_cicloe")

# ── Trips ─────────────────────────────────────────────────────────────
trips = pd.read_csv("data/viajes_ecobici_092026.csv")
trips["Ciclo_Estacion_Retiro"] = trips["Ciclo_Estacion_Retiro"].apply(fix_station_id)
trips["Ciclo_EstacionArribo"] = trips["Ciclo_EstacionArribo"].apply(fix_station_id)
trips = trips.dropna(subset=["Ciclo_Estacion_Retiro", "Ciclo_EstacionArribo"])
trips["Ciclo_Estacion_Retiro"] = trips["Ciclo_Estacion_Retiro"].astype(int)
trips["Ciclo_EstacionArribo"] = trips["Ciclo_EstacionArribo"].astype(int)

print(f"Stations: {len(stations)}  |  Trips: {len(trips):,}")
trips.head(3)

## 2. Clustering configuration

The **only** function you need to change to try a different clustering is `build_clusters()`.  
It must return a `pd.Series` mapping `station_id → cluster_label`.

Three ready-made strategies are provided:

| Strategy | Description |
|---|---|
| `cluster_by_kmeans(k)` | K-Means on lat/lon (geographic proximity) |
| `cluster_by_alcaldia()` | One cluster per *alcaldía* (borough) |
| `cluster_by_manual(mapping)` | User-supplied `{station_id: cluster}` dict |

In [ ]:
from sklearn.cluster import KMeans


# ── Strategy 1: K-Means on geographic coordinates ─────────────────────
def cluster_by_kmeans(stations_df: pd.DataFrame, k: int = 8, random_state: int = 42) -> pd.Series:
    """
    Cluster stations into `k` groups using K-Means on (lat, lon).
    Returns: Series  station_id → cluster_label (int 0..k-1)
    """
    coords = stations_df[["latitud", "longitud"]].values
    labels = KMeans(n_clusters=k, random_state=random_state, n_init=10).fit_predict(coords)
    return pd.Series(labels, index=stations_df.index, name="cluster")


# ── Strategy 2: cluster by alcaldía ───────────────────────────────────
def cluster_by_alcaldia(stations_df: pd.DataFrame) -> pd.Series:
    """
    One cluster per alcaldía. Label = integer code for each unique alcaldía.
    """
    codes, uniques = pd.factorize(stations_df["alcaldia"])
    mapping = pd.Series(codes, index=stations_df.index, name="cluster")
    print("Alcaldía → cluster:", dict(zip(uniques, range(len(uniques)))))
    return mapping


# ── Strategy 3: fully manual ──────────────────────────────────────────
def cluster_by_manual(stations_df: pd.DataFrame, mapping: dict) -> pd.Series:
    """
    Assign clusters from a user-supplied dict  {station_id: cluster_label}.
    Stations not in `mapping` are assigned to cluster -1.
    """
    return pd.Series(
        [mapping.get(sid, -1) for sid in stations_df.index],
        index=stations_df.index,
        name="cluster",
    )


# =====================================================================
# ◆  CHANGE THIS SINGLE LINE TO SWITCH CLUSTERING STRATEGY / PARAMS  ◆
# =====================================================================
station_clusters = cluster_by_kmeans(stations, k=8)
# station_clusters = cluster_by_alcaldia(stations)
# station_clusters = cluster_by_manual(stations, {1: 0, 2: 0, 3: 1, ...})
# =====================================================================

stations["cluster"] = station_clusters
print(f"Number of clusters: {stations['cluster'].nunique()}")
stations["cluster"].value_counts().sort_index()

## 3. Build the Markov-chain model

### Core idea

Every trip `(origin_station, destination_station)` falls into one of three categories **with respect to a given cluster $C$**:

| Category | Origin | Destination |
|---|---|---|
| **Internal** | $\in C$ | $\in C$ |
| **Outflow** | $\in C$ | $\notin C$ |
| **Inflow** | $\notin C$ | $\in C$ |

For each cluster we build:

* **Internal transition matrix $T$** — rows are origin stations in $C$, columns are destination stations in $C$, entries are transition *probabilities* (each row sums to the fraction of trips that stay internal; outflow accounts for the rest).  
* **Outflow matrix $O$** — for every station in $C$, the probability of the bike going to each *other cluster*.  
* **Inflow summary** — total bikes arriving from each external cluster.

In [ ]:
class ClusterMarkovModel:
    """
    Markov-chain model for bicycle flows within and between station clusters.

    Parameters
    ----------
    trips_df : DataFrame
        Must have columns 'Ciclo_Estacion_Retiro' and 'Ciclo_EstacionArribo'.
    stations_df : DataFrame
        Must have a 'cluster' column; index = station_id.
    """

    def __init__(self, trips_df: pd.DataFrame, stations_df: pd.DataFrame):
        self.stations = stations_df.copy()
        self.cluster_labels = sorted(self.stations["cluster"].unique())

        # Build a station → cluster lookup (includes only known stations)
        self._station_to_cluster = self.stations["cluster"].to_dict()

        # Annotate each trip with origin and destination cluster
        self.trips = trips_df.copy()
        self.trips["cluster_origin"] = (
            self.trips["Ciclo_Estacion_Retiro"].map(self._station_to_cluster)
        )
        self.trips["cluster_dest"] = (
            self.trips["Ciclo_EstacionArribo"].map(self._station_to_cluster)
        )
        # Drop trips where either station is unknown
        self.trips = self.trips.dropna(subset=["cluster_origin", "cluster_dest"])
        self.trips["cluster_origin"] = self.trips["cluster_origin"].astype(int)
        self.trips["cluster_dest"] = self.trips["cluster_dest"].astype(int)

        # Pre-compute per-cluster results
        self._internal_matrices: dict[int, pd.DataFrame] = {}
        self._outflow_matrices: dict[int, pd.DataFrame] = {}
        self._inflow_summaries: dict[int, pd.Series] = {}
        self._flow_counts: dict[int, dict] = {}

        for cl in self.cluster_labels:
            self._build_cluster(cl)

    # ── Private: build one cluster ────────────────────────────────────
    def _build_cluster(self, cluster_id: int):
        cl_stations = self.stations[self.stations["cluster"] == cluster_id].index

        # Trips originating in this cluster
        from_cl = self.trips[self.trips["cluster_origin"] == cluster_id]
        # Trips arriving to this cluster
        to_cl = self.trips[self.trips["cluster_dest"] == cluster_id]

        # ── Internal transitions (origin AND destination in cluster) ───
        internal = from_cl[from_cl["cluster_dest"] == cluster_id]
        internal_counts = (
            internal.groupby(["Ciclo_Estacion_Retiro", "Ciclo_EstacionArribo"])
            .size()
            .unstack(fill_value=0)
            .reindex(index=cl_stations, columns=cl_stations, fill_value=0)
        )

        # ── Outflow: trips leaving the cluster, grouped by dest cluster ─
        outflow = from_cl[from_cl["cluster_dest"] != cluster_id]
        outflow_counts = (
            outflow.groupby(["Ciclo_Estacion_Retiro", "cluster_dest"])
            .size()
            .unstack(fill_value=0)
            .reindex(index=cl_stations, fill_value=0)
        )
        # Ensure all other clusters appear as columns
        for other_cl in self.cluster_labels:
            if other_cl != cluster_id and other_cl not in outflow_counts.columns:
                outflow_counts[other_cl] = 0
        outflow_counts = outflow_counts[sorted(outflow_counts.columns)]

        # ── Build the row-stochastic transition matrix ────────────────
        # Each row = one origin station in the cluster.
        # Columns = [internal station destinations...] + ["_out_cluster_X" ...]
        # We normalise so each row sums to 1.
        combined = pd.concat(
            [internal_counts, outflow_counts.rename(columns=lambda c: f"_out_{c}")],
            axis=1,
        ).fillna(0)
        row_sums = combined.sum(axis=1)
        row_sums = row_sums.replace(0, 1)  # avoid division by zero for idle stations
        transition = combined.div(row_sums, axis=0)

        # Separate back into internal-only and outflow-only probability matrices
        int_cols = [c for c in transition.columns if not str(c).startswith("_out_")]
        out_cols = [c for c in transition.columns if str(c).startswith("_out_")]
        self._internal_matrices[cluster_id] = transition[int_cols]
        self._outflow_matrices[cluster_id] = transition[out_cols].rename(
            columns=lambda c: int(c.replace("_out_", ""))
        )

        # ── Inflow: bikes arriving from each external cluster ─────────
        inflow = to_cl[to_cl["cluster_origin"] != cluster_id]
        inflow_summary = inflow.groupby("cluster_origin").size()
        inflow_summary = inflow_summary.reindex(
            [c for c in self.cluster_labels if c != cluster_id], fill_value=0
        )
        self._inflow_summaries[cluster_id] = inflow_summary

        # ── Counts summary ────────────────────────────────────────────
        self._flow_counts[cluster_id] = {
            "total_departures": int(len(from_cl)),
            "internal_trips": int(len(internal)),
            "outflow_trips": int(len(outflow)),
            "inflow_trips": int(len(inflow)),
        }

    # ── Public API ────────────────────────────────────────────────────
    def internal_transition_matrix(self, cluster_id: int) -> pd.DataFrame:
        """Row-stochastic matrix of internal transitions (probabilities)."""
        return self._internal_matrices[cluster_id]

    def outflow_matrix(self, cluster_id: int) -> pd.DataFrame:
        """Per-station probability of leaving to each other cluster."""
        return self._outflow_matrices[cluster_id]

    def inflow_summary(self, cluster_id: int) -> pd.Series:
        """Total bike arrivals from each external cluster (counts)."""
        return self._inflow_summaries[cluster_id]

    def flow_counts(self, cluster_id: int) -> dict:
        """Dict with total_departures, internal_trips, outflow_trips, inflow_trips."""
        return self._flow_counts[cluster_id]

    def inter_cluster_matrix(self) -> pd.DataFrame:
        """
        Aggregate cluster-to-cluster flow matrix (counts).
        Entry (i, j) = number of trips from cluster i to cluster j.
        """
        mat = (
            self.trips.groupby(["cluster_origin", "cluster_dest"])
            .size()
            .unstack(fill_value=0)
            .reindex(index=self.cluster_labels, columns=self.cluster_labels, fill_value=0)
        )
        return mat

    def inter_cluster_transition_matrix(self) -> pd.DataFrame:
        """
        Row-stochastic cluster-to-cluster transition probabilities.
        """
        mat = self.inter_cluster_matrix().astype(float)
        row_sums = mat.sum(axis=1).replace(0, 1)
        return mat.div(row_sums, axis=0)

    def summary(self) -> pd.DataFrame:
        """One-row-per-cluster summary of trip flows."""
        rows = []
        for cl in self.cluster_labels:
            fc = self._flow_counts[cl]
            n_stations = (self.stations["cluster"] == cl).sum()
            rows.append({"cluster": cl, "n_stations": n_stations, **fc})
        return pd.DataFrame(rows).set_index("cluster")

In [ ]:
model = ClusterMarkovModel(trips, stations)
model.summary()

## 4. Inspect a single cluster

In [ ]:
# ◆ Choose a cluster to inspect
INSPECT_CLUSTER = 0

fc = model.flow_counts(INSPECT_CLUSTER)
print(f"=== Cluster {INSPECT_CLUSTER} ===")
print(f"  Total departures : {fc['total_departures']:>8,}")
print(f"  Internal trips   : {fc['internal_trips']:>8,}  ({fc['internal_trips']/max(fc['total_departures'],1)*100:.1f}%)")
print(f"  Outflow trips    : {fc['outflow_trips']:>8,}  ({fc['outflow_trips']/max(fc['total_departures'],1)*100:.1f}%)")
print(f"  Inflow trips     : {fc['inflow_trips']:>8,}")

In [ ]:
# Internal transition matrix (station-to-station within the cluster)
T = model.internal_transition_matrix(INSPECT_CLUSTER)
print(f"Internal transition matrix shape: {T.shape}")
T.head(10)

In [ ]:
# Outflow probabilities (station → external cluster)
O = model.outflow_matrix(INSPECT_CLUSTER)
print(f"Outflow matrix shape: {O.shape}")
O.head(10)

In [ ]:
# Inflow from other clusters (counts)
inf = model.inflow_summary(INSPECT_CLUSTER)
print("Inflow (bikes arriving from each external cluster):")
inf

## 5. Visualisations

In [ ]:
# ── 5a. Heatmap of the internal transition matrix ─────────────────────
T_full = model.internal_transition_matrix(INSPECT_CLUSTER)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(T_full.values, cmap="YlOrRd", aspect="auto")
ax.set_title(f"Internal transition probabilities — Cluster {INSPECT_CLUSTER}", fontsize=13)
ax.set_xlabel("Destination station")
ax.set_ylabel("Origin station")
plt.colorbar(im, ax=ax, label="Probability")
plt.tight_layout()
plt.show()

In [ ]:
# ── 5b. Cluster-to-cluster flow heatmap ──────────────────────────────
inter = model.inter_cluster_transition_matrix()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(inter.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(inter.columns)))
ax.set_xticklabels(inter.columns)
ax.set_yticks(range(len(inter.index)))
ax.set_yticklabels(inter.index)
ax.set_xlabel("Destination cluster")
ax.set_ylabel("Origin cluster")
ax.set_title("Cluster-to-cluster transition probabilities")
plt.colorbar(im, ax=ax, label="Probability")

# Annotate cells
for i in range(len(inter.index)):
    for j in range(len(inter.columns)):
        val = inter.iloc[i, j]
        color = "white" if val > 0.4 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=color)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5c. Bar chart: internal vs outflow vs inflow per cluster ─────────
summary_df = model.summary()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(summary_df))
w = 0.25
ax.bar(x - w, summary_df["internal_trips"], width=w, label="Internal")
ax.bar(x,     summary_df["outflow_trips"],  width=w, label="Outflow")
ax.bar(x + w, summary_df["inflow_trips"],   width=w, label="Inflow")
ax.set_xticks(x)
ax.set_xticklabels([f"C{c}" for c in summary_df.index])
ax.set_xlabel("Cluster")
ax.set_ylabel("Trip count")
ax.set_title("Internal / Outflow / Inflow trips per cluster")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 5d. Folium map coloured by cluster ───────────────────────────────
cmap_colors = plt.cm.tab10.colors

center = [stations["latitud"].mean(), stations["longitud"].mean()]
m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")

for sid, row in stations.iterrows():
    cl = int(row["cluster"])
    color = to_hex(cmap_colors[cl % len(cmap_colors)])
    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.8,
        popup=f"Station {sid} — Cluster {cl}",
    ).add_to(m)

m

## 6. Quick-swap: try a different number of clusters

Just re-run the two cells below with a different `k` to rebuild everything.

In [ ]:
# ◆ Change k here and re-run this cell + the next one
NEW_K = 12

stations["cluster"] = cluster_by_kmeans(stations, k=NEW_K)
model = ClusterMarkovModel(trips, stations)
model.summary()

In [ ]:
# Re-draw the map with the new clusters
center = [stations["latitud"].mean(), stations["longitud"].mean()]
m2 = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")

cmap20 = plt.cm.tab20.colors
for sid, row in stations.iterrows():
    cl = int(row["cluster"])
    color = to_hex(cmap20[cl % len(cmap20)])
    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.8,
        popup=f"Station {sid} — Cluster {cl}",
    ).add_to(m2)

m2